In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from pyspark.sql import functions as F
from lh import common, silver

cfg = common.load_config("dev")
silver.ensure_tables(spark, cfg)
cat = cfg["catalog"]
spec = {"key": ["id"], "order": "last_modified", "mode": "scd2",
        "columns": {"id": "string", "city": "city", "phone": "phone", "last_modified": "timestamp"}}

def add(rows):                                        # append rows to a small bronze test table
    (spark.createDataFrame(rows, "id string, city string, phone string, last_modified string, ts string")
          .withColumn("_ingest_ts", F.to_timestamp("ts")).withColumn("_business_date", F.to_date("ts"))
          .withColumn("_source_file", F.lit("t10.csv")).drop("ts")
          .write.mode("append").saveAsTable(f"{cat}.bronze.t10_test"))

def clean_up():
    spark.sql(f"DROP TABLE IF EXISTS {cat}.bronze.t10_test")
    spark.sql(f"DROP TABLE IF EXISTS {cat}.silver.t10_test")
    spark.sql(f"DELETE FROM {cat}.ops.dq_results WHERE table_name = 't10_test'")
    spark.sql(f"DELETE FROM {cat}.ops.silver_state WHERE table_name = 't10_test'")

clean_up()
steps = []
add([("K1", "BLR", "7937946724", "2026-09-01 10:00:00", "2026-09-02 00:30:00"),
     ("K1", "Bangalore", "+91-7937946724", "2026-09-02 10:00:00", "2026-09-02 00:30:00"),   # same after cleaning
     ("K1", "Bombay", "7937946724", "2026-09-03 10:00:00", "2026-09-02 00:30:00"),
     ("K2", "Delhi", "7937946725", "2026-09-01 10:00:00", "2026-09-02 00:30:00")])
steps.append(silver.load_scd2(spark, cfg, "t10_test", spec))
add([("K1", "Delhi", "7937946724", "2026-08-30 10:00:00", "2026-09-05 00:30:00"),            # older version, late
     ("K1", "Madras", "7937946724", "2026-09-04 10:00:00", "2026-09-05 00:30:00"),
     ("K3", "hyd", "7937946726", "2026-09-04 10:00:00", "2026-09-05 00:30:00")])
steps.append(silver.load_scd2(spark, cfg, "t10_test", spec))
steps.append(silver.load_scd2(spark, cfg, "t10_test", spec))                                  # nothing new
history = [(r["city"], r["is_current"]) for r in
           spark.table(f"{cat}.silver.t10_test").filter("id = 'K1'").orderBy("valid_from").collect()]
display(spark.table(f"{cat}.silver.t10_test").select("id", "city", "valid_from", "valid_to", "is_current").orderBy("id", "valid_from"))
clean_up()
got = [(s["rows"], s["inserted"], s["updated"]) for s in steps]
print(got, history)
print("PASS" if got == [(4, 3, 0), (3, 3, 1), (0, 0, 0)] and history ==
      [("Delhi", False), ("Bengaluru", False), ("Mumbai", False), ("Chennai", True)] else "FAIL")